# 06 — Walk-Forward Backtest (God Mode v3)Simulate the full trading strategy using walk-forward test predictions.Supports both XGBoost Stage 2 and direct TCN+GRU (bypass) modes.

In [ ]:
!pip install -q xgboost ccxt PyWavelets hmmlearn numba scikit-learn pyyaml tqdm pyarrow matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json

REPO_DIR = '/content/scalp2_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} fetch && git -C {REPO_DIR} reset --hard origin/main
else:
    !git clone https://github.com/umutergul74/Scalp2.git {REPO_DIR}

sys.path.insert(0, REPO_DIR)

import logging
logging.basicConfig(level=logging.INFO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

from scalp2.config import load_config
from scalp2.execution.trade_manager import TradeManager, TradeState, TradeStatus

config = load_config(f'{REPO_DIR}/config.yaml')
DATA_DIR = '/content/drive/MyDrive/scalp2/data/processed'

print(f'bypass_xgboost = {getattr(config.model, "bypass_xgboost", False)}')

In [ ]:
# Load data + predictions
df = pd.read_parquet(f'{DATA_DIR}/BTC_USDT_labeled.parquet')

bypass = getattr(config.model, 'bypass_xgboost', False)
pkl_file = 'wf_predictions_stage1.pkl' if bypass else 'wf_predictions.pkl'
print(f'Loading: {pkl_file}')

with open(f'{DATA_DIR}/{pkl_file}', 'rb') as f:
    wf_predictions = pickle.load(f)

print(f'Loaded {len(df):,} bars, {len(wf_predictions)} folds')
print(f'First fold test: bars {wf_predictions[0]["test_start"]}-{wf_predictions[0]["test_end"]}')
print(f'Last fold test: bars {wf_predictions[-1]["test_start"]}-{wf_predictions[-1]["test_end"]}')
first_date = df.index[wf_predictions[0]['test_start']]
last_date = df.index[min(wf_predictions[-1]['test_end']-1, len(df)-1)]
print(f'Date range: {first_date} -> {last_date}')

In [ ]:
# ============================================================
#  PURE SIGNAL DIAGNOSTIC — No SL/TP, just hold for N bars
#  Respects direction_filter from config
# ============================================================
import warnings
warnings.filterwarnings('ignore')

seq_len = config.model.seq_len
HOLD_BARS = 10
CONF_THRESHOLD_DIAG = 0.40
DIR_FILTER = getattr(config.execution, 'direction_filter', 'both')

close = df['close'].values
diag_trades = []

for fold_data in wf_predictions:
    fold_idx = fold_data['fold_idx']
    test_start = fold_data['test_start']
    preds = fold_data['test_probabilities']
    n_preds = len(preds)
    pred_offset = test_start + seq_len

    for i in range(n_preds):
        bar = pred_offset + i
        exit_bar = bar + HOLD_BARS
        if exit_bar >= len(df):
            break

        p_arr = preds[i]
        cls = int(np.argmax(p_arr))
        if cls == 1:  # Hold
            continue

        # Direction filter
        if DIR_FILTER == 'long_only' and cls == 0:  # Skip SHORT
            continue
        if DIR_FILTER == 'short_only' and cls == 2:  # Skip LONG
            continue

        conf = max(p_arr[0], p_arr[2])
        if conf < CONF_THRESHOLD_DIAG:
            continue

        entry_price = close[bar]
        exit_price = close[exit_bar]
        raw_ret = (exit_price - entry_price) / entry_price

        if cls == 0:  # Short
            raw_ret = -raw_ret

        diag_trades.append({
            'fold': fold_idx,
            'bar': bar,
            'direction': 'SHORT' if cls == 0 else 'LONG',
            'confidence': conf,
            'raw_return': raw_ret,
            'timestamp': df.index[bar],
        })

diag_df = pd.DataFrame(diag_trades)

if len(diag_df) == 0:
    print('No diagnostic trades!')
else:
    n = len(diag_df)
    mean_ret = diag_df['raw_return'].mean()
    median_ret = diag_df['raw_return'].median()
    wr = (diag_df['raw_return'] > 0).mean()
    wins = diag_df.loc[diag_df['raw_return'] > 0, 'raw_return']
    losses = diag_df.loc[diag_df['raw_return'] < 0, 'raw_return']
    pf = abs(wins.sum() / losses.sum()) if len(losses) > 0 else float('inf')
    sharpe = mean_ret / diag_df['raw_return'].std() * np.sqrt(252 * 24 / HOLD_BARS) if diag_df['raw_return'].std() > 0 else 0

    RT_COST = 2 * (2 + 2) / 10_000
    net_ret = mean_ret - RT_COST

    print('=' * 70)
    print(f'  PURE SIGNAL TEST -- Hold {HOLD_BARS} bars, conf >= {CONF_THRESHOLD_DIAG}')
    print(f'  Direction filter: {DIR_FILTER}')
    print(f'  No SL, No TP, No TradeManager -- raw model signal only')
    print('=' * 70)
    print(f'  Trades:           {n:,}')
    print(f'  Win Rate:         {wr*100:.1f}%')
    print(f'  Profit Factor:    {pf:.4f}')
    print(f'  Mean Return:      {mean_ret*100:.4f}% (GROSS, no costs)')
    print(f'  Median Return:    {median_ret*100:.4f}%')
    print(f'  RT Cost:          {RT_COST*100:.4f}%')
    print(f'  Net Return/trade: {net_ret*100:.4f}% (after 8bps RT cost)')
    print(f'  Annualized Sharpe:{sharpe:.2f}')
    print(f'  Cumulative GROSS: {diag_df["raw_return"].sum()*100:.2f}%')
    print(f'  Cumulative NET:   {(diag_df["raw_return"].sum() - RT_COST*n)*100:.2f}%')
    print()

    # By confidence bucket
    diag_df['conf_bucket'] = pd.cut(diag_df['confidence'], bins=[0.4, 0.5, 0.6, 0.7, 0.8, 1.0])
    bucket_stats = diag_df.groupby('conf_bucket', observed=True).agg(
        n=('raw_return', 'count'),
        mean_ret=('raw_return', 'mean'),
        win_rate=('raw_return', lambda x: (x > 0).mean()),
    )
    bucket_stats['mean_ret'] = bucket_stats['mean_ret'] * 100
    bucket_stats['win_rate'] = bucket_stats['win_rate'] * 100
    print('  Confidence Bucket Analysis:')
    print(bucket_stats.to_string())
    print()

    # By direction
    dir_stats = diag_df.groupby('direction').agg(
        n=('raw_return', 'count'),
        mean_ret=('raw_return', 'mean'),
        win_rate=('raw_return', lambda x: (x > 0).mean()),
    )
    dir_stats['mean_ret'] = dir_stats['mean_ret'] * 100
    dir_stats['win_rate'] = dir_stats['win_rate'] * 100
    print('  Direction Analysis:')
    print(dir_stats.to_string())
    print()

    # By year
    diag_df['year'] = pd.to_datetime(diag_df['timestamp']).dt.year
    yr = diag_df.groupby('year').agg(
        n=('raw_return', 'count'),
        gross_pnl=('raw_return', 'sum'),
        win_rate=('raw_return', lambda x: (x > 0).mean()),
    )
    yr['gross_pnl'] = yr['gross_pnl'] * 100
    yr['net_pnl'] = yr['gross_pnl'] - yr['n'] * RT_COST * 100
    yr['win_rate'] = yr['win_rate'] * 100
    print('  Yearly Breakdown (GROSS vs NET):')
    print(yr.to_string())


In [ ]:
# ============================================================
#  PURE SIGNAL MATCH BACKTEST ENGINE (Full Pyramiding, No Early Exit)
# ============================================================
from tqdm import tqdm

seq_len = config.model.seq_len
exec_cfg = config.execution
label_cfg = config.labeling
order_cfg = config.execution.order_execution

DIR_FILTER = getattr(exec_cfg, 'direction_filter', 'both')
HOLD_BARS = label_cfg.max_holding_bars  # 10 bars
CONF_THRESHOLD = exec_cfg.confidence_threshold
LEVERAGE = exec_cfg.position_sizing.leverage
POSITION_SIZE = 0.10  # 10% base equity per tranche
MAX_CONCURRENT_TRANCHES = 20  # Practically infinite for 10-bar hold
MAX_TRADES_PER_DAY = 24

# Costs
MAKER_FEE_BPS = order_cfg.maker_fee_bps
SLIPPAGE_BPS = order_cfg.slippage_bps
FLAT_RT_COST = 2 * (SLIPPAGE_BPS + MAKER_FEE_BPS) / 10_000

print(f'Strategy: True Pure Signal Match (Pyramiding up to {MAX_CONCURRENT_TRANCHES})')
print(f'Direction: {DIR_FILTER} | Hold {HOLD_BARS} bars | Conf >= {CONF_THRESHOLD}')
print(f'Leverage: {LEVERAGE}x | Size/Tranche: {POSITION_SIZE*100:.0f}% | RT Cost: {FLAT_RT_COST*100:.4f}%')

all_trades = []
cumulative_pnl = 0.0
skip_reasons = {}
fold_stats = []

def _close_tranche(t, exit_price, status, exit_bar, exit_timestamp, fold_idx):
    global cumulative_pnl
    entry_price = t['entry_price']
    direction = t['direction']
    if direction == 'LONG':
        raw_ret = (exit_price - entry_price) / entry_price
    else:
        raw_ret = (entry_price - exit_price) / entry_price

    net = (raw_ret - FLAT_RT_COST) * POSITION_SIZE * LEVERAGE
    cumulative_pnl += net

    all_trades.append({
        'fold': fold_idx,
        'direction': direction,
        'entry_price': entry_price,
        'exit_price': exit_price,
        'bars_held': t['bars_held'],
        'status': status,
        'gross_pnl': raw_ret * POSITION_SIZE * LEVERAGE,
        'net_pnl': net,
        'cost': FLAT_RT_COST * POSITION_SIZE * LEVERAGE,
        'position_size': POSITION_SIZE,
        'entry_bar': t['entry_bar'],
        'exit_bar': exit_bar,
        'timestamp': exit_timestamp,
    })

for fold_data in tqdm(wf_predictions, desc='Backtesting folds'):
    fold_idx = fold_data['fold_idx']
    test_start = fold_data['test_start']
    test_end = fold_data['test_end']
    preds = fold_data['test_probabilities']
    n_preds = len(preds)
    pred_offset = test_start + seq_len

    fold_skips = {}
    fold_trades = 0

    active_tranches = []
    pending_tranches = []
    
    daily_count = 0
    prev_date = None

    for i in range(n_preds):
        bar = pred_offset + i
        if bar >= len(df):
            break

        row = df.iloc[bar]
        cur_date = row.name.date() if hasattr(row.name, 'date') else None
        if cur_date != prev_date:
            daily_count = 0
            prev_date = cur_date

        # -- Activate pending tranches --
        still_pending = []
        for pt in pending_tranches:
            if pt['entry_bar'] == bar:
                pt['entry_price'] = row['open']
                active_tranches.append(pt)
            else:
                still_pending.append(pt)
        pending_tranches = still_pending

        # -- Evaluate Exits for Active Tranches --
        surviving_tranches = []
        for t in active_tranches:
            t['bars_held'] += 1
            if t['bars_held'] >= HOLD_BARS:
                # TIME EXIT
                _close_tranche(t, row['close'], 'TIME_EXIT', bar, row.name, fold_idx)
            else:
                surviving_tranches.append(t)
        active_tranches = surviving_tranches

        # Get model's prediction
        p_arr = preds[i]
        cls = int(np.argmax(p_arr))
        conf = max(p_arr[0], p_arr[2])

        # -- Evaluate Entry --
        if cls == 1:
            fold_skips['hold'] = fold_skips.get('hold', 0) + 1
            continue

        if DIR_FILTER == 'long_only' and cls == 0:
            fold_skips['short_blocked'] = fold_skips.get('short_blocked', 0) + 1
            continue
        if DIR_FILTER == 'short_only' and cls == 2:
            fold_skips['long_blocked'] = fold_skips.get('long_blocked', 0) + 1
            continue

        if conf < CONF_THRESHOLD:
            fold_skips['low_conf'] = fold_skips.get('low_conf', 0) + 1
            continue

        total_active_pending = len(active_tranches) + len(pending_tranches)
        if total_active_pending >= MAX_CONCURRENT_TRANCHES:
            fold_skips['max_tranches'] = fold_skips.get('max_tranches', 0) + 1
            continue

        if daily_count >= MAX_TRADES_PER_DAY:
            fold_skips['daily_cap'] = fold_skips.get('daily_cap', 0) + 1
            continue

        next_bar = bar + 1
        if next_bar >= len(df):
            break

        pending_tranches.append({
            'direction': 'LONG' if cls == 2 else 'SHORT',
            'entry_bar': next_bar,
            'bars_held': 0,
        })
        daily_count += 1
        fold_trades += 1

    # Force close at fold end
    last_bar = min(test_end - 1, len(df) - 1)
    last_row = df.iloc[last_bar]
    for t in active_tranches:
        _close_tranche(t, last_row['close'], 'FOLD_END', last_bar, last_row.name, fold_idx)

    for k, v in fold_skips.items():
        skip_reasons[k] = skip_reasons.get(k, 0) + v

    start_date = df.index[min(pred_offset, len(df)-1)]
    end_date = df.index[min(test_end-1, len(df)-1)]
    fold_stats.append({
        'fold': fold_idx,
        'start': str(start_date)[:10],
        'end': str(end_date)[:10],
        'bars': min(n_preds, len(df) - pred_offset),
        'trades': fold_trades,
        'top_skip': max(fold_skips, key=fold_skips.get) if fold_skips else 'none',
        'top_skip_n': max(fold_skips.values()) if fold_skips else 0,
    })

trades_df = pd.DataFrame(all_trades)
fold_stats_df = pd.DataFrame(fold_stats)

print(f'\nTotal trades: {len(trades_df)}')
print(f'Cumulative PnL ({LEVERAGE}x): {cumulative_pnl*100:.2f}%')
print(f'Skip reasons: {skip_reasons}')


In [ ]:
# ============================================================
#  PER-FOLD DIAGNOSTICS
# ============================================================
print('=' * 80)
print('  FOLD-BY-FOLD BREAKDOWN')
print('=' * 80)
print(fold_stats_df.to_string(index=False))

# Folds with zero trades
zero_folds = fold_stats_df[fold_stats_df['trades'] == 0]
trade_folds = fold_stats_df[fold_stats_df['trades'] > 0]
print(f'\nFolds with trades: {len(trade_folds)} / {len(fold_stats_df)}')
print(f'Folds with ZERO trades: {len(zero_folds)} / {len(fold_stats_df)}')

if len(zero_folds) > 0:
    print('\nTop skip reasons in zero-trade folds:')
    print(zero_folds['top_skip'].value_counts().to_string())

# Year coverage
if len(trades_df) > 0:
    trades_df['timestamp'] = pd.to_datetime(trades_df['timestamp'])
    trades_df['year'] = trades_df['timestamp'].dt.year
    print(f'\nTrades per year:')
    print(trades_df.groupby('year').size().to_string())

In [ ]:
# ============================================================
#  RESULTS
# ============================================================
if len(trades_df) == 0:
    print("No trades!")
else:
    net = trades_df['net_pnl'].values
    n = len(trades_df)
    wins = net[net > 0]
    losses = net[net < 0]
    wr = len(wins) / n
    pf = abs(wins.sum() / losses.sum()) if len(losses) else float('inf')

    status_counts = trades_df['status'].value_counts()

    print('=' * 60)
    print(f'  Total trades: {n}')
    print(f'  Win rate: {wr*100:.1f}%')
    print(f'  Profit factor: {pf:.4f}')
    print(f'  Expectancy: {net.mean()*100:.4f}%')
    print(f'  Avg win: +{wins.mean()*100:.4f}%' if len(wins) else '  No wins')
    print(f'  Avg loss: {losses.mean()*100:.4f}%' if len(losses) else '  No losses')
    print(f'  Avg bars held: {trades_df["bars_held"].mean():.1f}')
    print(f'  Gross PnL: {trades_df["gross_pnl"].sum()*100:.2f}%')
    print(f'  Net PnL: {net.sum()*100:.2f}%')
    print(f'  Cost impact: {(trades_df["gross_pnl"].sum()-net.sum())*100:.2f}%')
    print()
    print('  Close reasons:')
    for s, c in status_counts.items():
        print(f'    {s:20s}: {c:5d} ({c/n*100:.1f}%)')
    print('=' * 60)

    # Equity curve
    fig, ax = plt.subplots(figsize=(16, 5))
    trades_df['cum_pnl'] = trades_df['net_pnl'].cumsum() * 100
    ax.plot(trades_df['timestamp'], trades_df['cum_pnl'], linewidth=1)
    ax.axhline(0, color='grey', linestyle='--', alpha=0.5)
    ax.set_title(f'Equity Curve ({LEVERAGE}x)')
    ax.set_ylabel('Cumulative PnL (%)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Yearly
    yearly = trades_df.groupby('year').agg(
        trades=('net_pnl', 'count'),
        net_pnl=('net_pnl', 'sum'),
        win_rate=('net_pnl', lambda x: (x > 0).mean()),
    )
    yearly['net_pnl'] = yearly['net_pnl'] * 100
    yearly['win_rate'] = yearly['win_rate'] * 100
    print('\nYearly:')
    print(yearly.to_string())